# Compare policy performance

Reads a simulation log (one row per epoch x day x product, written by the policy notebooks)
and summarises each epoch and product.

In [1]:
import polars as pl
from pathlib import Path

# Relative to this notebook (code/sim/) -> results/ at the repo root.
RESULTS = Path('../../results')
LOG_FILE = RESULTS / 'Newsvendor_SingleSupplier_DP_SinglePrice.parquet'

sim_log = pl.read_parquet(LOG_FILE)
sim_log.head()

epoch,seed,day,product,demand,sold,price,wastage,order_qty,order_cost,stock
i32,i64,i32,str,i64,i64,f64,i64,i64,f64,i64
0,0,0,"""FOODS_1""",1305,1259,2.18,0,null,null,69362
0,0,0,"""FOODS_2""",1073,1024,2.8,0,null,null,95552
0,0,0,"""FOODS_3""",7602,6294,2.16,0,61000,22570.0,53075
0,0,0,"""HOBBIES_1""",1798,1461,2.87,0,null,null,668159
0,0,0,"""HOBBIES_2""",36,31,1.73,0,null,null,34518


## Performance per epoch and product

* **revenue** = units sold x the price they sold at
* **stockout_days** = days that end with no stock on hand
* **lost_units** on a stockout day = expected buyers - units sold. Expected buyers = demand x
  the share of customers who bought at that price on days the product was in stock (same epoch,
  product and price; falls back to the epoch-product share if that price was never in stock).
* **underage_cost** = lost_units x unit margin, where unit margin = price - average unit cost of
  that epoch's orders (floored at 0).

In [2]:
def performance(log):
    """Totals per (epoch, product) from a daily simulation log."""
    log = log.with_columns(
        (pl.col('stock') == 0).alias('stockout'),
        (pl.col('sold') * pl.col('price')).alias('revenue'),
    )

    # Share of demand that buys at a price, measured on in-stock days (demand there is not
    # capped by stock). Per epoch/product/price, with an epoch/product fallback.
    in_stock = log.filter(~pl.col('stockout') & (pl.col('demand') > 0))
    share_at_price = in_stock.group_by('epoch', 'product', 'price').agg(
        (pl.col('sold').sum() / pl.col('demand').sum()).alias('share_price')
    )
    share_overall = in_stock.group_by('epoch', 'product').agg(
        (pl.col('sold').sum() / pl.col('demand').sum()).alias('share_overall')
    )

    # Average unit cost of everything ordered in the epoch (null if nothing was ordered).
    unit_cost = log.group_by('epoch', 'product').agg(
        (pl.col('order_cost').sum() / pl.col('order_qty').sum()).alias('unit_cost')
    )

    shortfall = (
        log.filter(pl.col('stockout'))
        .join(share_at_price, on=['epoch', 'product', 'price'], how='left')
        .join(share_overall, on=['epoch', 'product'], how='left')
        .join(unit_cost, on=['epoch', 'product'], how='left')
        .with_columns(
            (pl.col('demand') * pl.coalesce('share_price', 'share_overall') - pl.col('sold'))
            .clip(lower_bound=0).alias('lost_units'),
            (pl.col('price') - pl.col('unit_cost').fill_null(0)).clip(lower_bound=0).alias('unit_margin'),
        )
        .group_by('epoch', 'product')
        .agg(
            pl.col('lost_units').sum(),
            (pl.col('lost_units') * pl.col('unit_margin')).sum().alias('underage_cost'),
        )
    )

    return (
        log.group_by('epoch', 'product')
        .agg(
            pl.col('demand').sum().alias('total_demand'),
            pl.col('sold').sum().alias('total_sold'),
            pl.col('revenue').sum().alias('total_revenue'),
            pl.col('wastage').sum().alias('total_wastage'),
            pl.col('order_qty').sum().alias('total_order_qty'),      # nulls (no order) skipped
            pl.col('order_cost').sum().alias('total_cost'),
            pl.col('stockout').sum().alias('stockout_days'),
        )
        .join(shortfall, on=['epoch', 'product'], how='left')
        .with_columns(pl.col('lost_units', 'underage_cost').fill_null(0))  # never out of stock
        .sort('epoch', 'product')
    )


perf = performance(sim_log)
perf

epoch,product,total_demand,total_sold,total_revenue,total_wastage,total_order_qty,total_cost,stockout_days,lost_units,underage_cost
i32,str,i64,i64,f64,i64,i64,f64,u32,f64,f64
0,"""FOODS_1""",2541283,1546061,3.7918e6,9181,1571500,487350.0,24,18021.61361,38924.563858
0,"""FOODS_2""",2002499,1229218,3.9794e6,7000,1173000,566270.0,11,4771.721256,13299.962783
0,"""FOODS_3""",12475477,9440530,2.0027e7,1127231,10589100,3.895525e6,52,196767.807863,345220.164951
0,"""HOBBIES_1""",2763083,1518756,5.1879e6,0,892500,394350.0,0,0.0,0.0
0,"""HOBBIES_2""",227900,121449,254326.82,3082,100000,27700.0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…
99,"""FOODS_3""",12543597,9733961,2.0539e7,874005,10606680,3969642.1,58,209155.5443,379950.912817
99,"""HOBBIES_1""",2768660,1518925,5.1934e6,0,900000,405750.0,0,0.0,0.0
99,"""HOBBIES_2""",226320,121165,253200.46,2341,130000,40900.0,0,0.0,0.0


## Mean and standard deviation over epochs, per product

Each metric becomes two columns, `<metric>_mean` and `<metric>_std`, side by side. The std is
the spread across epochs (sample std, ddof=1).

In [3]:
metrics = [c for c in perf.columns if c not in ('epoch', 'product')]


def mean_std(df, by):
    """Mean and std across epochs of every metric, grouped by `by`."""
    return (
        df.group_by(by)
        .agg(
            *[pl.col(m).mean().alias(f'{m}_mean') for m in metrics],
            *[pl.col(m).std().alias(f'{m}_std') for m in metrics],
        )
        .select(*by, *[f'{m}_{s}' for m in metrics for s in ('mean', 'std')])   # mean next to std
        .sort(by[::-1])
    )


summary = mean_std(perf, ['product'])
summary

product,total_demand_mean,total_demand_std,total_sold_mean,total_sold_std,total_revenue_mean,total_revenue_std,total_wastage_mean,total_wastage_std,total_order_qty_mean,total_order_qty_std,total_cost_mean,total_cost_std,stockout_days_mean,stockout_days_std,lost_units_mean,lost_units_std,underage_cost_mean,underage_cost_std
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""FOODS_1""",2.5503e6,5370.751681,1611601.3,53903.569416,3.9181e6,90613.920433,5517.21,3288.37713,1.609065e6,59598.388859,507744.75,19593.369864,7.38,8.080129,5364.913859,5714.187337,11301.749023,12093.902355
"""FOODS_2""",2004525.2,6189.524446,1.2354e6,9110.78546,3.9937e6,24388.793681,5508.71,2417.170816,1.16977e6,15788.2804,589900.0,17987.587583,8.1,8.408149,5016.6087,5151.105604,13237.692164,13919.697038
"""FOODS_3""",1.2497e7,27288.330069,9.6027e6,430882.533291,2.0326e7,743508.77859,899209.57,109891.69886,1.0498e7,408841.329804,3.9080e6,150316.498885,39.28,7.483423,151957.347796,33723.808961,272377.412938,58876.473424
"""HOBBIES_1""",2.7682e6,5197.752827,1.5385e6,19211.214261,5.2240e6,35177.772135,0.0,0.0,898080.0,30593.186011,417587.4,23249.145151,1.89,5.002817,1328.62299,3663.41722,4422.63173,12256.704428
"""HOBBIES_2""",226987.01,941.009103,121969.49,2697.722086,253605.01,3578.407147,6233.39,3942.900473,113864.0,9046.641433,31628.54,4342.280103,14.58,17.784444,844.869387,1033.624737,1614.066334,1993.353254
"""HOUSEHOLD_1""",4347166.3,14883.767804,2.8619e6,31150.120579,1.0613e7,81572.448859,0.0,0.0,2.362e6,60038.708053,1314252.9,36121.112053,6.87,6.404946,9601.956328,9556.95816,30905.958053,30816.484897
"""HOUSEHOLD_2""",1.5863e6,3372.532353,1.0907e6,11327.767275,5.4620e6,45313.623915,0.0,0.0,861040.0,23902.968836,626949.4,18730.618687,5.36,8.586482,3061.327639,5082.616943,13113.331858,21727.40173


## Save to Excel

Both tables, rounded to 2 decimals, in one workbook in `results/`: sheet `per_epoch` (per epoch
and product) and sheet `summary` (mean and std over epochs).

In [4]:
import polars.selectors as cs
from xlsxwriter import Workbook

excel_file = RESULTS / f'performance_{LOG_FILE.stem}.xlsx'
with Workbook(excel_file) as wb:
    perf.with_columns(cs.float().round(2)).write_excel(wb, worksheet='per_epoch')
    summary.with_columns(cs.float().round(2)).write_excel(wb, worksheet='summary')
print(f'saved {excel_file.resolve()}')

saved C:\Users\prade\OneDrive\Documents\Code\retail-revenue-management\results\performance_Newsvendor_SingleSupplier_DP_SinglePrice.xlsx


## Compare every policy in results/

The policy name is taken from the file name.

In [ ]:
all_perf = pl.concat([
    performance(pl.read_parquet(f)).with_columns(pl.lit(f.stem).alias('policy'))
    for f in sorted(RESULTS.glob('*.parquet'))
])

comparison = mean_std(all_perf, ['policy', 'product'])   # sorted by product, then policy
comparison